# MLFlow AutoLogging and Tracking Server

In this notebook, we will set up an MLFlow tracking server and enable autologging for a machine learning model. We will use the `mlflow` library to log parameters, metrics, and artifacts during model training.

### Start MLFlow Tracking Server
To start the MLFlow tracking server, you can run the following command in your terminal:

```bash
mlflow ui
```

This will start the MLFlow UI at `http://127.0.0.1:5000`.

### Set MLFlow Tracking URI
Now we will set the tracking URI in our notebook to point to the MLFlow server we just started. This allows us to log our experiments and models to the server.

```python
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
```

In [1]:
!pip install ucimlrepo
!pip install "azureml-mlflow==1.59.0"

In [2]:
import azureml.mlflow

In [3]:
azureml.mlflow.__version__

'1.59.0'

In [4]:
# Set MLFlow Tracking URI (Not necessary in Azure ML)
import mlflow
# mlflow.set_tracking_uri("http://127.0.0.1:5000")

In [5]:
mlflow.__version__

'2.15.1'

In [6]:
from ucimlrepo import fetch_ucirepo
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Fetch Statlog (German Credit Data) dataset
credit_data = fetch_ucirepo(id=27)

# Features and targets
X = credit_data.data.features
y = credit_data.data.targets

# Categorical encoding for X and y
X = pd.get_dummies(X)
le = LabelEncoder()
y = le.fit_transform(y)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

/anaconda/envs/azureml_py38/lib/python3.10/site-packages/sklearn/preprocessing/_label.py:114: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


## 1. Basic MLFlow Usage

In this example, we log a RandomForestClassifier training run to the MLflow tracking server. We create a new experiment, log model parameters and test accuracy, add a descriptive tag, and save the trained model with its input/output signature. This ensures our experiment is fully tracked and reproducible in the MLflow UI.

In [7]:
# Print the MLflow run URL for easy access
tracking_url = mlflow.get_tracking_uri()
print(tracking_url)

azureml://westeurope.api.azureml.ms/mlflow/v1.0/subscriptions/b20ef8b9-237f-48d7-ad34-9e0564fe09f0/resourceGroups/pulze_beluga_rg/providers/Microsoft.MachineLearningServices/workspaces/beluga


In [9]:
from mlflow.models import infer_signature

# Set experiment name for grouping related runs
mlflow.set_experiment("Credit Risk Prediction Experiment")

# Start an MLflow run
with mlflow.start_run() as run:
    # Define model hyperparameters
    n_estimators = 100
    max_depth = 6
    random_state = 42

    # Log hyperparameters for reproducibility
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_param("random_state", random_state)

    # Train the model
    rf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=random_state
    )
    rf.fit(X_train, y_train)

    # Evaluate and log metrics
    score = rf.score(X_test, y_test)
    mlflow.log_metric("test_accuracy", score)
    mlflow.log_metric("score", score)
    print(f"Test Accuracy: {score:.4f}")

    # Add descriptive tags for this run
    mlflow.set_tag("Training Info", "Basic RandomForest model for credit risk")
    mlflow.set_tag("model_type", "RandomForestClassifier")
    mlflow.set_tag("dataset", "Statlog (German Credit Data)")
    mlflow.set_tag("author", "your_name")  # TODO Replace with your name or team
    mlflow.set_tag("purpose", "baseline model")
    mlflow.set_tag("feature_engineering", "one-hot encoding, label encoding")
    mlflow.set_tag("notes", "No hyperparameter tuning, default split")

    # Infer and log model signature for input/output schema
    signature = infer_signature(X_train, rf.predict(X_train))

    # Log the trained model
    model_info = mlflow.sklearn.log_model(
        rf,
        artifact_path="classifier",
        signature=signature
    )

    # Optionally, log a sample input/output as an artifact for clarity
    sample_input = X_test.iloc[:5]
    sample_output = rf.predict(sample_input)
    sample_df = sample_input.copy()
    sample_df["prediction"] = sample_output
    sample_df.to_csv("sample_predictions.csv", index=False)
    mlflow.log_artifact("sample_predictions.csv")

    # Print the MLflow run URL for easy access
    tracking_url = mlflow.get_tracking_uri()
    print(f"Run details: {tracking_url}/#/experiments/{run.info.experiment_id}/runs/{run.info.run_id}")


Test Accuracy: 0.8696
Run details: azureml://westeurope.api.azureml.ms/mlflow/v1.0/subscriptions/b20ef8b9-237f-48d7-ad34-9e0564fe09f0/resourceGroups/pulze_beluga_rg/providers/Microsoft.MachineLearningServices/workspaces/beluga/#/experiments/b1a31339-54c9-4ffd-9505-a260f77dc160/runs/852f3750-431e-42cd-802b-31ae1566f306


/anaconda/envs/azureml_py38/lib/python3.10/site-packages/mlflow/types/utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/02/19 10:53:03 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - mlflow (current: 2.19.0, required: mlflow==2.15.1)
To fix the mism

In [10]:
model_info.model_uri

'runs:/852f3750-431e-42cd-802b-31ae1566f306/classifier'

In [11]:
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)

y_pred = loaded_model.predict(X_test)

from sklearn.metrics import accuracy_score
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test Accuracy: {test_accuracy:.4f}")


KeyboardInterrupt: 

## 2. Autologging

In this section, we will enable autologging for our machine learning model using MLFlow. Autologging automatically logs parameters, metrics, and artifacts without requiring explicit logging calls in the code.


In [12]:
mlflow.autolog()

rf = RandomForestClassifier(n_estimators=100, max_depth=6)
rf.fit(X_train, y_train)

2026/02/19 11:00:13 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/02/19 11:00:14 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/02/19 11:00:14 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.
2026/02/19 11:00:14 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'd657729b-0547-4e94-bc1a-6c1b76ab95f1', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/02/19 11:00:14 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/anaconda/envs/azureml_py38/lib/python3.10/site-packages/mlflow/types/utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will 

RandomForestClassifier(max_depth=6)

## 3. Autologging with GridSearchCV

In [13]:
import mlflow.sklearn

mlflow.sklearn.autolog(
   log_input_examples=True,
   log_model_signatures=True
)

parameters = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 6, 9]
}

from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(RandomForestClassifier(), parameters, cv=3)
grid_search.fit(X_train, y_train)

2026/02/19 11:04:11 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '357c2160-3236-49b6-9031-a7fc0b20665e', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
2026/02/19 11:04:11 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/anaconda/envs/azureml_py38/lib/python3.10/site-packages/mlflow/types/utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Mi

GridSearchCV(cv=3, estimator=RandomForestClassifier(),
             param_grid={'max_depth': [3, 6, 9],
                         'n_estimators': [100, 200, 300]})

## 4. Hyperparameter Tuning with hyperopt

In [14]:
import mlflow
import mlflow.sklearn
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import numpy as np

# Enable autologging
mlflow.sklearn.autolog(log_models=True, silent=False)

# ---------------------------------------------------------------------
# Objective function
# ---------------------------------------------------------------------
def objective(params):
    n_estimators = int(params["n_estimators"])
    max_depth = int(params["max_depth"])

    with mlflow.start_run(nested=True):
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42
        )
        model.fit(X_train, y_train)

        return {"loss": -model.score(X_test, y_test), "status": STATUS_OK}


# ---------------------------------------------------------------------
# Hyperopt search space
# ---------------------------------------------------------------------
search_space = {
    "n_estimators": hp.quniform("n_estimators", 50, 300, 10),
    "max_depth": hp.quniform("max_depth", 3, 20, 1),
}

# ---------------------------------------------------------------------
# Main MLflow run: evaluate best model WITHOUT manual metric logging
# ---------------------------------------------------------------------
trials = Trials()

with mlflow.start_run(run_name="rf_hyperopt_main"):

    # Run hyperparameter optimization
    best_params = fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=10,
        trials=trials,
        rstate=np.random.default_rng(42),
    )

    # Convert float to int
    best_params = {k: int(v) for k, v in best_params.items()}
    print("Best hyperparameters:", best_params)

    # TODO Train final model using best parameters
    final_model = RandomForestClassifier(
            n_estimators=best_params["n_estimators"],
            max_depth=best_params["max_depth"],
            random_state=42
        )
    final_model.fit(X_train, y_train)

    # Final evaluation
    final_pred = final_model.predict(X_test)
    final_prob = final_model.predict_proba(X_test)[:, 1]

    accuracy_score(y_test, final_pred)
    precision_score(y_test, final_pred)
    # TODO implement new classification metrics

    # Autolog will save the final model automatically

2026/02/19 11:12:45 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/anaconda/envs/azureml_py38/lib/python3.10/site-packages/mlflow/types/utils.py:406: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."

2026/02/19 11:12:49 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/anaconda/envs/azureml_py38/lib/python3.

 50%|█████     | 5/10 [00:55<00:55, 11.01s/trial, best loss: -0.8731884057971014]
Best hyperparameters: {'max_depth': 7, 'n_estimators': 150}


In [ ]:
import mlflow
import mlflow.sklearn
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)
import numpy as np
import matplotlib.pyplot as plt

# Enable autologging (params, models, artifacts; metrics captured via post-training metric calls)
mlflow.sklearn.autolog(log_models=True, silent=False)

# ---------------------------------------------------------------------
# Objective function
# ---------------------------------------------------------------------
def objective(params):
    n_estimators = int(params["n_estimators"])
    max_depth = int(params["max_depth"])

    with mlflow.start_run(nested=True):
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42
        )
        model.fit(X_train, y_train)

        # Post-training metric autologging (optional inside trials)
        # accuracy_score(y_test, model.predict(X_test))
        # model.score(X_test, y_test)

        return {"loss": -model.score(X_test, y_test), "status": STATUS_OK}


# ---------------------------------------------------------------------
# Hyperopt search space
# ---------------------------------------------------------------------
search_space = {
    "n_estimators": hp.quniform("n_estimators", 50, 300, 10),
    "max_depth": hp.quniform("max_depth", 3, 20, 1),
}

# ---------------------------------------------------------------------
# Main MLflow run: evaluate best model WITHOUT manual metric logging
# ---------------------------------------------------------------------
trials = Trials()

with mlflow.start_run(run_name="rf_hyperopt_main"):

    # Run hyperparameter optimization
    best_params = fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=10,
        trials=trials,
        rstate=np.random.default_rng(42),
    )

    # Convert float to int
    best_params = {k: int(v) for k, v in best_params.items()}
    print("Best hyperparameters:", best_params)

    # -----------------------------------------------------------------
    # TODO 1: Train final model using best parameters
    # -----------------------------------------------------------------
    # Replace `None` with a properly initialized estimator using best_params, then fit.
    final_model = RandomForestClassifier(
            n_estimators=best_params['n_estimators'],
            max_depth=best_params['max_depth'],
            random_state=42
        )
    final_model.fit(X_train, y_train)

    # -----------------------------------------------------------------
    # Final evaluation
    # -----------------------------------------------------------------
    final_pred = final_model.predict(X_test)
    final_prob = final_model.predict_proba(X_test)[:, 1]

    # Trigger MLflow post-training metric autologging by calling sklearn metric APIs:
    accuracy_score(y_test, final_pred)
    # TODO 2: Implement additional classification metrics (captured automatically by MLflow)
    # Hint: Call metric calculations to have MLflow autolog them:


    # -----------------------------------------------------------------
    # TODO 3: Feature importance logging
    # -----------------------------------------------------------------
    # Goal: Log a bar plot (PNG) of top-K important features + a CSV with all importances.
    # Steps:
    
    #  3.1) Extract importances from final_model.feature_importances_
    importances = [] # TODO
    
    #  3.2) Build a DataFrame with columns: ["feature", "importance"] using X_train.columns
    fi_df = pd.DataFrame({"feature": X_train.columns, "importance": importances})
    fi_df = fi_df.sort_values("importance", ascending=False).reset_index(drop=True)
    
    #  3.3) Create a horizontal bar plot for top-K features (e.g., K=20) and save to file
    top_k = 20
    top_df = fi_df.head(top_k)
    plt.figure(figsize=(9, 6))
    plt.barh(top_df["feature"][::-1], top_df["importance"][::-1], color="#1f77b4")
    plt.title("Top Feature Importances")
    plt.xlabel("Importance")
    plt.ylabel("Feature")
    plt.tight_layout()
    fig_path = "feature_importance.png"
    plt.savefig(fig_path, dpi=150); plt.close()
    
    #  3.4) Save full CSV with all importances
    csv_path = "feature_importance.csv"
    fi_df.to_csv(csv_path, index=False)
    
    #  3.5) Log both artifacts to MLflow
    # mlflow.log_...(...) TODO

    # -----------------------------------------------------------------
    # TODO 4: Model registration in MLflow Model Registry
    # -----------------------------------------------------------------
    # Goal: Register the final_model under a registry name and (optionally) transition to "Staging".
    # Steps:
    #  4.1) Choose a registry name, e.g.:
    #       registry_name = "GermanCredit_Scoring"
    registry_name = "GermanCredit_Scoring"
    
    #  4.2) Log & register the model in one step:
    #       mlflow.sklearn.log_model(
    #           sk_model=<TODO>,
    #           artifact_path="final_model",
    #           registered_model_name=<TODO>
    #       )
